# Deutsch-Jozsa Algorithm

Extend constant-versus-balanced classification to four input qubits. The algorithm assumes the function is promised to be either constant or balanced.

Run the cells from top to bottom in a fresh Python kernel. See the [repository README](../README.md) for environment setup.

**Bit ordering:** Qiskit displays bit strings with the highest-index bit on the left and bit 0 on the right. Ideal simulator results are used here; shot counts can fluctuate when several outcomes have nonzero probability.

In [ ]:
# Imports needed to run this notebook independently.
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

## Classifying a randomly selected oracle

The balanced oracle computes the parity of the input bits. An ideal constant oracle returns `0000`; this particular parity oracle returns `1111`. The oracle is chosen randomly on each run.

In [ ]:
import random
def constant0(n):
    qc = QuantumCircuit(n + 1)
    return qc

def constant1(n):
    qc = QuantumCircuit(n + 1)
    qc.x(n)
    return qc

def balanced(n):
    qc = QuantumCircuit(n + 1)
    for x in range(n):
        qc.cx(x , n)
    return qc 

n = 4

types = ["constant0", "constant1", "balanced"]
choice = random.choice(types)
print("Selected oracle:", choice)

if choice == "constant0": ch = constant0(n)
elif choice == "constant1": ch = constant1(n)
else: ch = balanced(n)

qc = QuantumCircuit(n + 1, n)
qc.x(n)
qc.h(range(n + 1))
qc.compose(ch, inplace=True)
qc.h(range(n))
qc.measure(range(n), range(n))

display(qc.draw("mpl"))

sm = Aer.get_backend("qasm_simulator")
ts = transpile(qc, sm)
rlt = sm.run(ts, shots = 1024).result().get_counts(qc)
print(rlt)
display(plot_histogram(rlt))

if "0"*n in rlt and rlt["0"*n] == 1024:
    print("Constant")
else:
    print("Balanced")